**Configuración de GPU**

In [ ]:
import torch
import os

# Verificación de GPU
print("="*70)
print(" Configuración de GPU")
print("="*70)
print(f"PyTorch: {torch.__version__}")
print(f"CUDA disponible: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memoria GPU: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print(" WARNING: No se detectó GPU. El entrenamiento será lento.")

print("="*70)

In [ ]:
!pip install -q scikit-learn
!pip install -q timm

**Configuración de Parámetros**

In [ ]:
class EvalConfig:
    """Configuración para evaluación"""

    # Rutas
    DATA_PATH = '/ruta_a_dataset'
    BACKBONE_PATH = '/ruta_a_backbone'
    REID_CSV = '/ruta_a_split'
    OUTPUT_DIR = '/output/eval_results'

    # Modelo
    BACKBONE = 'resnet50' # Opciones: resnet50, tiny_vit
    BATCH_SIZE = 128
    NUM_WORKERS = 2

    # Evaluación
    TOP_K = [1, 5, 10, 20]
    KNN_K = 10
    MIN_IMAGES_PER_ID = 2

    # Visualización
    N_VISUALIZE = 10
    TSNE_SAMPLES = 1000

    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

eval_config = EvalConfig()
os.makedirs(eval_config.OUTPUT_DIR, exist_ok=True)

print(" Configuración de evaluación lista")

**Dataset de Evaluación Re-id**

In [ ]:
import os
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as T
import pandas as pd
import numpy as np
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics.pairwise import cosine_similarity, euclidean_distances
from sklearn.neighbors import KNeighborsClassifier
from sklearn.manifold import TSNE
import json
import random

class ReIDDataset(Dataset):
    """Dataset para re-identificación"""

    def __init__(self, df, root_dir, transform=None):
        self.root_dir = root_dir
        self.paths = df["filename"].tolist()
        self.labels = df["label"].tolist() if "label" in df.columns else [0] * len(df)
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        rel_path = self.paths[idx]

        full_path = os.path.join(self.root_dir, rel_path)

        if not os.path.exists(full_path):
            raise FileNotFoundError(f"Imagen no encontrada: {full_path}")

        img = Image.open(full_path).convert("RGB")

        if self.transform:
            img = self.transform(img)

        return img, self.labels[idx], rel_path

**Cargar Modelo**

In [ ]:
import timm

def load_trained_backbone(backbone_path, backbone_type='resnet50', device='cuda'):
    """Carga el backbone entrenado con DINO"""

    print(f"\n Cargando modelo desde: {backbone_path}")

    # Construir arquitectura
    if backbone_type == 'resnet50':
        model = torchvision.models.resnet50(weights=None)
        model.fc = nn.Identity()
    
    elif backbone_type == 'tiny_vit':
        model = timm.create_model(
            'tiny_vit_5m_224',
            pretrained=False,
            num_classes=0
        )
    
    else:
        raise ValueError(f"Arquitectura no soportada: {backbone_type}")

    # Cargar pesos
    state_dict = torch.load(backbone_path, map_location=device)
    model.load_state_dict(state_dict)
    model.eval()
    model.to(device)

    print(f" Modelo cargado: {backbone_type}")

    return model

**Extracción de Características**

In [ ]:
def extract_features(model, dataloader, device='cuda', normalize=True):
    """
    Extrae features (embeddings) de todas las imágenes

    Returns:
        features: numpy array de shape (N, dim_features)
        labels: numpy array de shape (N,)
        paths: lista de rutas de imágenes
    """

    print("\n Extrayendo features...")

    all_features = []
    all_labels = []
    all_paths = []

    model.eval()

    with torch.no_grad():
        for imgs, labels, paths in tqdm(dataloader, desc="Extrayendo"):
            imgs = imgs.to(device)

            # Forward pass
            features = model(imgs)

            # Normalizar (importante para similitud coseno)
            if normalize:
                features = nn.functional.normalize(features, dim=1)

            all_features.append(features.cpu().numpy())
            all_labels.extend(labels.numpy())
            all_paths.extend(paths)

    # Concatenar todo
    features = np.vstack(all_features)
    labels = np.array(all_labels)

    print(f" Features extraídos: {features.shape}")
    print(f"   - Dimensión: {features.shape[1]}")
    print(f"   - Total imágenes: {features.shape[0]}")
    print(f"   - Identidades únicas: {len(np.unique(labels))}")

    return features, labels, all_paths

**Querys**

In [ ]:
def create_query_gallery_split(df, min_images=2, query_per_id=1):
    """
    Crea split de query y gallery

    Query: 1 imagen por identidad
    Gallery: El resto de imágenes

    Solo incluye identidades con al menos min_images
    """

    print(f"\n Creando split Query/Gallery...")

    # Filtrar identidades con suficientes imágenes
    id_counts = df['label'].value_counts()
    valid_ids = id_counts[id_counts >= min_images].index
    df_filtered = df[df['label'].isin(valid_ids)].copy()

    print(f"   - Identidades válidas: {len(valid_ids)} (con >={min_images} imgs)")
    print(f"   - Total imágenes válidas: {len(df_filtered)}")

    # Crear columna de split
    splits = []
    query_indices = []

    for label in valid_ids:
        label_indices = df_filtered[df_filtered['label'] == label].index.tolist()

        # Elegir query aleatoriamente
        query_idx = random.sample(label_indices, query_per_id)
        query_indices.extend(query_idx)

    # Marcar splits
    df_filtered['split'] = 'gallery'
    df_filtered.loc[query_indices, 'split'] = 'query'

    n_query = (df_filtered['split'] == 'query').sum()
    n_gallery = (df_filtered['split'] == 'gallery').sum()

    print(f"   - Query: {n_query} imágenes ({len(valid_ids)} identidades)")
    print(f"   - Gallery: {n_gallery} imágenes")

    return df_filtered

**Métricas de Evaluación**

In [ ]:
def compute_cmc_map(query_features, query_labels, gallery_features, gallery_labels,
                    top_k=[1, 5, 10, 20]):
    """
    Calcula métricas de re-identificación:
    - CMC (Cumulative Matching Characteristic) @ K
    - mAP (mean Average Precision)

    Args:
        query_features: (N_q, dim)
        query_labels: (N_q,)
        gallery_features: (N_g, dim)
        gallery_labels: (N_g,)
        top_k: Lista de valores K para Rank-K accuracy
    """

    print("\n Calculando métricas...")

    # Calcular matriz de similitud (cosine)
    similarity_matrix = cosine_similarity(query_features, gallery_features)

    # Para cada query, ordenar gallery por similitud (mayor a menor)
    indices = np.argsort(-similarity_matrix, axis=1)

    # CMC
    cmc = np.zeros(len(gallery_labels))
    average_precisions = []

    for i in range(len(query_labels)):
        query_label = query_labels[i]
        sorted_labels = gallery_labels[indices[i]]

        # Encontrar posiciones donde hay match
        matches = (sorted_labels == query_label)

        # CMC: encontrar primera posición correcta
        if matches.any():
            first_match_idx = np.where(matches)[0][0]
            cmc[first_match_idx:] += 1

        # Average Precision
        n_relevant = matches.sum()
        if n_relevant > 0:
            precision = np.cumsum(matches) / (np.arange(len(matches)) + 1)
            ap = (precision * matches).sum() / n_relevant
            average_precisions.append(ap)
        else:
            average_precisions.append(0)

    # Normalizar CMC
    cmc = cmc / len(query_labels)

    # mAP
    mAP = np.mean(average_precisions)

    # Extraer Rank-K accuracies
    results = {"mAP": mAP}
    for k in top_k:
        if k <= len(cmc):
            results[f"Rank-{k}"] = cmc[k-1]

    return results, cmc

def evaluate_knn(query_features, query_labels, gallery_features, gallery_labels, k=10):
    """Evalúa con k-NN classifier"""

    print(f"\n Evaluando k-NN (k={k})...")

    knn = KNeighborsClassifier(n_neighbors=k, metric='cosine')
    knn.fit(gallery_features, gallery_labels)
    accuracy = knn.score(query_features, query_labels)

    return accuracy

In [ ]:
def plot_cmc_curve(cmc, top_k=100, save_path=None):
    """Grafica la curva CMC"""

    plt.figure(figsize=(10, 6))
    ranks = np.arange(1, min(top_k, len(cmc)) + 1)
    plt.plot(ranks, cmc[:top_k], linewidth=2, color='#2E86AB')
    plt.xlabel('Rank', fontsize=12)
    plt.ylabel('Matching Rate', fontsize=12)
    plt.title('Cumulative Matching Characteristic (CMC)', fontsize=14, fontweight='bold')
    plt.grid(True, alpha=0.3)
    plt.xlim([1, top_k])
    plt.ylim([0, 1])
    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f" CMC guardado: {save_path}")

    plt.show()

def visualize_query_results(query_idx, query_features, query_labels, query_paths,
                           gallery_features, gallery_labels, gallery_paths,
                           top_k=5, save_path=None):
    """
    Visualiza los top-K resultados más similares para una query
    """

    # Calcular similitud de esta query con toda la gallery
    query_feat = query_features[query_idx].reshape(1, -1)
    similarities = cosine_similarity(query_feat, gallery_features)[0]

    # Ordenar por similitud
    sorted_indices = np.argsort(-similarities)

    query_label = query_labels[query_idx]
    query_path = query_paths[query_idx]

    # Plot
    fig, axes = plt.subplots(1, top_k + 1, figsize=(3 * (top_k + 1), 3))

    # Query image
    query_img = Image.open(query_path).convert("RGB")
    axes[0].imshow(query_img)
    axes[0].set_title(f"Query\nID: {query_label}", fontsize=10, fontweight='bold')
    axes[0].axis('off')
    axes[0].spines['top'].set_color('red')
    axes[0].spines['bottom'].set_color('red')
    axes[0].spines['left'].set_color('red')
    axes[0].spines['right'].set_color('red')
    axes[0].spines['top'].set_linewidth(3)
    axes[0].spines['bottom'].set_linewidth(3)
    axes[0].spines['left'].set_linewidth(3)
    axes[0].spines['right'].set_linewidth(3)

    # Top-K results
    for i in range(top_k):
        gallery_idx = sorted_indices[i]
        gallery_path = gallery_paths[gallery_idx]
        gallery_label = gallery_labels[gallery_idx]
        sim = similarities[gallery_idx]

        img = Image.open(gallery_path).convert("RGB")
        axes[i + 1].imshow(img)

        # Color: verde si match, rojo si no
        color = 'green' if gallery_label == query_label else 'red'
        axes[i + 1].set_title(f"Rank {i+1}\nID: {gallery_label}\nSim: {sim:.3f}",
                             fontsize=9, color=color)
        axes[i + 1].axis('off')

        # Borde de color
        for spine in axes[i + 1].spines.values():
            spine.set_edgecolor(color)
            spine.set_linewidth(2)

    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=200, bbox_inches='tight')

    plt.show()

def plot_tsne_embeddings(features, labels, n_samples=1000, save_path=None):
    """Visualiza embeddings con t-SNE"""

    print(f"\n Generando visualización t-SNE...")

    # Samplear si hay muchas imágenes
    if len(features) > n_samples:
        indices = np.random.choice(len(features), n_samples, replace=False)
        features_sample = features[indices]
        labels_sample = labels[indices]
    else:
        features_sample = features
        labels_sample = labels

    # t-SNE
    tsne = TSNE(n_components=2, random_state=42, perplexity=30)
    embeddings_2d = tsne.fit_transform(features_sample)

    # Plot
    plt.figure(figsize=(12, 10))

    # Colorear por identidad (solo algunas para no saturar)
    unique_labels = np.unique(labels_sample)
    n_colors = min(20, len(unique_labels))
    selected_labels = np.random.choice(unique_labels, n_colors, replace=False)

    for label in selected_labels:
        mask = labels_sample == label
        plt.scatter(embeddings_2d[mask, 0], embeddings_2d[mask, 1],
                   label=f'ID {label}', alpha=0.6, s=50)

    # Resto en gris
    other_mask = ~np.isin(labels_sample, selected_labels)
    plt.scatter(embeddings_2d[other_mask, 0], embeddings_2d[other_mask, 1],
               color='gray', alpha=0.2, s=20, label='Otros')

    plt.title('t-SNE de Embeddings DINO', fontsize=14, fontweight='bold')
    plt.xlabel('Dimensión 1', fontsize=12)
    plt.ylabel('Dimensión 2', fontsize=12)
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f" t-SNE guardado: {save_path}")

    plt.show()

**Iniciar Evaluación**

In [ ]:
def evaluate_reid(config):
    """Pipeline completo de evaluación"""

    # Cargar datos
    print("\nCargando datos de re-identificación...")
    df = pd.read_csv(config.REID_CSV)

    if not {"filename", "label"}.issubset(df.columns):
        raise ValueError("El CSV debe contener: filename, label")

    print(f"   - Total imágenes: {len(df)}")
    print(f"   - Identidades: {df['label'].nunique()}")

    # Crear split query/gallery
    df = create_query_gallery_split(df, min_images=config.MIN_IMAGES_PER_ID)

    # Preparar transformaciones
    transform = T.Compose([
        T.Resize(256),
        T.CenterCrop(224),
        T.ToTensor(),
        T.Normalize(mean=[0.485, 0.456, 0.406],
                   std=[0.229, 0.224, 0.225]),
    ])

    # Crear dataset y dataloader
    dataset = ReIDDataset(
        df,
        root_dir=config.DATA_PATH,
        transform=transform
    )

    dataloader = DataLoader(
        dataset,
        batch_size=config.BATCH_SIZE,
        shuffle=False,
        num_workers=config.NUM_WORKERS,
        pin_memory=True
    )

    # Cargar modelo
    model = load_trained_backbone(
        config.BACKBONE_PATH,
        backbone_type=config.BACKBONE,
        device=config.DEVICE
    )

    # Extraer features
    features, labels, paths = extract_features(model, dataloader, config.DEVICE)

    # Separar query y gallery
    query_mask = df['split'] == 'query'
    gallery_mask = df['split'] == 'gallery'

    query_features = features[query_mask]
    query_labels = labels[query_mask]
    query_paths = [p for i, p in enumerate(paths) if query_mask.iloc[i]]

    gallery_features = features[gallery_mask]
    gallery_labels = labels[gallery_mask]
    gallery_paths = [p for i, p in enumerate(paths) if gallery_mask.iloc[i]]

    print(f"\nSplit completado:")
    print(f"   - Query: {len(query_features)}")
    print(f"   - Gallery: {len(gallery_features)}")

    # Calcular métricas
    results, cmc = compute_cmc_map(
        query_features, query_labels,
        gallery_features, gallery_labels,
        top_k=config.TOP_K
    )

    # k-NN
    knn_acc = evaluate_knn(
        query_features, query_labels,
        gallery_features, gallery_labels,
        k=config.KNN_K
    )
    results['kNN_accuracy'] = knn_acc

    # Imprimir resultados
    print("\n" + "="*70)
    print("RESULTADOS DE RE-IDENTIFICACIÓN")
    print("="*70)
    print(f"mAP: {results['mAP']:.4f}")
    for k in config.TOP_K:
        if f"Rank-{k}" in results:
            print(f"Rank-{k}: {results[f'Rank-{k}']:.4f}")
    print(f"k-NN Accuracy (k={config.KNN_K}): {results['kNN_accuracy']:.4f}")
    print("="*70)

    # 11. Guardar resultados
    results_path = os.path.join(config.OUTPUT_DIR, 'results.json')
    with open(results_path, 'w') as f:
        json.dump(results, f, indent=4)
    print(f"\nResultados guardados: {results_path}")

    # Visualizaciones

    # CMC curve
    plot_cmc_curve(cmc, top_k=100,
                   save_path=os.path.join(config.OUTPUT_DIR, 'cmc_curve.png'))

    # Visualizar queries de ejemplo
    print(f"\nVisualizando {config.N_VISUALIZE} queries de ejemplo...")
    for i in range(min(config.N_VISUALIZE, len(query_features))):
        visualize_query_results(
            i, query_features, query_labels, query_paths,
            gallery_features, gallery_labels, gallery_paths,
            top_k=5,
            save_path=os.path.join(config.OUTPUT_DIR, f'query_{i+1:03d}.png')
        )

    # t-SNE
    plot_tsne_embeddings(
        features, labels,
        n_samples=config.TSNE_SAMPLES,
        save_path=os.path.join(config.OUTPUT_DIR, 'tsne_embeddings.png')
    )

    print("\nEvaluación completada")
    print(f"Resultados en: {config.OUTPUT_DIR}")

    return results

In [ ]:
def export_features(features, labels, paths, save_path):
    """Exporta features a archivo .npz para uso posterior"""

    np.savez_compressed(
        save_path,
        features=features,
        labels=labels,
        paths=paths
    )
    print(f"Features exportados: {save_path}")

def load_features(load_path):
    """Carga features previamente guardados"""

    data = np.load(load_path, allow_pickle=True)
    return data['features'], data['labels'], data['paths']

print("\n" + "="*70)
print("INICIANDO EVALUACIÓN")
print("="*70)

# Evaluar
results = evaluate_reid(eval_config)

print("\nEVALUACIÓN COMPLETADA")
print(f"Resultados guardados en: {eval_config.OUTPUT_DIR}")

In [ ]:
import json

# Cargar resultados
results_path = os.path.join(eval_config.OUTPUT_DIR, 'results.json')
with open(results_path, 'r') as f:
    results = json.load(f)

# Mostrar tabla bonita
print("\n" + "="*70)
print("RESULTADOS FINALES - RE-IDENTIFICACIÓN DE PERROS")
print("="*70)
print(f"{'Métrica':<25} {'Valor':>10}")
print("-"*70)
print(f"{'mAP':<25} {results['mAP']:>10.4f}")
for k in [1, 5, 10, 20]:
    key = f'Rank-{k}'
    if key in results:
        print(f"{key:<25} {results[key]:>10.4f}")
print(f"{'k-NN Accuracy (k=10)':<25} {results['kNN_accuracy']:>10.4f}")
print("="*70)

# Comparación con baseline
print("\nInterpretación:")
print(f"  - mAP de {results['mAP']:.2%}: ", end="")
if results['mAP'] > 0.50:
    print("Excelente!")
elif results['mAP'] > 0.30:
    print("Bueno")
else:
    print("Necesita mejora")

print(f"  - Rank-1 de {results['Rank-1']:.2%}: ", end="")
if results['Rank-1'] > 0.60:
    print("Muy bueno para ReID")
elif results['Rank-1'] > 0.40:
    print("Aceptable")
else:
    print("Necesita mejora")